# 센서 투자 ROI 시나리오 (보고서 제4장 <표 9>·<표 10>·<그림 1>, 파일럿 판정 기준)

연간 절감액 S = N[(r0 − r1)Ci + (R1 − R0)·p·Co] − M, 회수 기간 T = I / S

- 입력값은 보고서 <표 9>의 가정이다. 센서·DAQ 단가(I)는 견적 확보 전 가정치다
- 시나리오의 r1·R1은 외부 데이터 성능을 그대로 옮긴 값이 아니라 범위를 보기 위한 가정이다

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

P = dict(N=300_000, Ci=100, p=0.02, M=1_000_000, I=12_000_000, r0=0.50, R0=0.92)
scen = pd.DataFrame([
    ('낙관 (외부 데이터 성능 재현)', 0.10, 0.995, 50_000),
    ('중립', 0.20, 0.97, 30_000),
    ('검사비 절감만 인정', 0.10, 0.92, 0),
    ('보수', 0.30, 0.95, 10_000),
    ('효과 없음', 0.50, 0.92, 0),
], columns=['시나리오', 'r1', 'R1', 'Co'])
scen['부품당 절감 (원)'] = (P['r0'] - scen.r1) * P['Ci'] + (scen.R1 - P['R0']) * P['p'] * scen.Co
scen['연간 절감 S (만 원)'] = (P['N'] * scen['부품당 절감 (원)'] - P['M']) / 1e4
scen['회수 기간 T (개월)'] = [P['I'] / (s * 1e4) * 12 if s > 0 else float('nan') for s in scen['연간 절감 S (만 원)']]
scen.round({'부품당 절감 (원)': 1, '연간 절감 S (만 원)': 0, '회수 기간 T (개월)': 1})

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
y = range(len(scen))[::-1]
a1.barh(list(y), scen['연간 절감 S (만 원)'], color=['tab:red' if v < 0 else 'grey' for v in scen['연간 절감 S (만 원)']])
a1.axvline(0, color='black', lw=0.8); a1.set_xlabel('연간 절감액 (만 원)')
a1.set_yticks(list(y)); a1.set_yticklabels(scen['시나리오'])
a2.barh(list(y), scen['회수 기간 T (개월)'].fillna(0), color='tab:blue')
a2.axvline(12, ls='--', color='grey'); a2.set_xlabel('회수 기간 (개월), 점선 = 1년')
for yi, t in zip(y, scen['회수 기간 T (개월)']):
    a2.text(0.5 if pd.isna(t) else t + 0.3, yi, '회수 불가' if pd.isna(t) else f'{t:.1f}', va='center')
fig.tight_layout()

## 파일럿 판정 기준: 허용 회수 기간에서 역산

검사비 절감만 인정(R1 = R0)하면 r1 ≤ r0 − (I/T + M) / (N·Ci)

In [ ]:
rows = []
for months in [12, 24]:
    need = P['I'] / (months / 12) + P['M']
    r1_max = P['r0'] - need / (P['N'] * P['Ci'])
    rows.append({'허용 회수 기간 (개월)': months, '필요 연간 절감 (만 원)': (need - P['M']) / 1e4,
                 '필요 검사율 r1 (재현율 0.92 유지)': f'{r1_max:.1%}', '현행 대비 감소 (%p)': round((P['r0'] - r1_max) * 100, 1)})
pd.DataFrame(rows)